# 02 — Prompt comparison & iteration

Compares four prompt strategies on the same 30 synthetic frontal chest X-rays,
using cached MedGemma 4B outputs (offline batch inference).

| version | intent | outcome |
|---|---|---|
| **baseline** | simple prompting | balanced but weak (opacity recall 40%) |
| **improved (v1)** | strict artifact-awareness | too permissive → max false negatives |
| **improved_v2** | strong safety bias | too strict → max false positives |
| **improved_v3** | balanced, symmetric rules | best on every metric |

The arc baseline → v1 → v2 → v3 is a measured engineering iteration.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

CLASSES = ['normal', 'suspected_opacity', 'uncertain']
CACHE = Path('../eval/cached_predictions')
VERSIONS = {'baseline':'baseline', 'v1 (improved)':'improved',
            'v2':'improved_v2', 'v3':'improved_v3'}

def load(mode):
    recs = json.loads((CACHE / f'predictions_{mode}.json').read_text())
    return [(r['ground_truth'], r['prediction']['predicted_class'],
             float(r['prediction']['confidence'])) for r in recs]

data = {label: load(mode) for label, mode in VERSIONS.items()}

In [ ]:
def metrics(rows):
    n = len(rows)
    acc = sum(gt==pr for gt,pr,_ in rows)/n
    f1s=[]
    for c in CLASSES:
        tp=sum(1 for gt,pr,_ in rows if gt==c and pr==c)
        fp=sum(1 for gt,pr,_ in rows if gt!=c and pr==c)
        fn=sum(1 for gt,pr,_ in rows if gt==c and pr!=c)
        p=tp/(tp+fp) if tp+fp else 0; r=tp/(tp+fn) if tp+fn else 0
        f1s.append(2*p*r/(p+r) if p+r else 0)
    opac_recall = (sum(1 for gt,pr,_ in rows if gt=='suspected_opacity' and pr=='suspected_opacity')
                   / sum(1 for gt,_,_ in rows if gt=='suspected_opacity'))
    fn_opac = sum(1 for gt,pr,_ in rows if gt=='suspected_opacity' and pr=='normal')
    return {'accuracy':acc, 'macro_f1':sum(f1s)/len(f1s),
            'opacity_recall':opac_recall, 'false_negatives':fn_opac,
            'uncertain_rate':sum(pr=='uncertain' for _,pr,_ in rows)/n}

df = pd.DataFrame({label: metrics(rows) for label, rows in data.items()}).T
df

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
for col, style in [('accuracy','-o'), ('macro_f1','-s'), ('opacity_recall','-^')]:
    ax.plot(df.index, df[col], style, label=col)
ax.set_ylim(0,1.05); ax.set_ylabel('score'); ax.grid(alpha=.3)
ax.set_title('Prompt iteration: baseline -> v1 -> v2 -> v3')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
def confusion(rows):
    m = np.zeros((3,3), int)
    for gt,pr,_ in rows:
        m[CLASSES.index(gt)][CLASSES.index(pr)] += 1
    return m

fig, axes = plt.subplots(1,4, figsize=(18,4))
for ax,(label,rows) in zip(axes, data.items()):
    m = confusion(rows)
    im = ax.imshow(m, cmap='Blues', vmin=0, vmax=10)
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels([c[:6] for c in CLASSES], rotation=45)
    ax.set_yticklabels([c[:6] for c in CLASSES])
    ax.set_title(label); ax.set_xlabel('pred'); ax.set_ylabel('truth')
    for i in range(3):
        for j in range(3):
            ax.text(j,i,m[i][j],ha='center',va='center',
                    color='white' if m[i][j]>5 else 'black')
plt.tight_layout(); plt.show()

## Reading the result

- **v1** collapsed opacity recall to 0%: the strict wording let the model explain findings
  away into confident `normal` (confidence 0.70 -> 0.95 on missed opacities).
- **v2** overcorrected: a blanket safety bias flagged almost every image (normal precision 0%).
- **v3** applies balanced rules: doubt -> `uncertain`, but clean images may still be `normal`.
  It beats baseline on every metric and cuts false negatives from 6 to 1.

Lesson: a plausible 'safety' prompt silently made the system more dangerous; only measurement
revealed it. Fluency is not clinical correctness.